# Strike — research notes

Thomas Ou · November 2026

Strike labels short MMA video windows as **strike attempt** or **no attempt**. A committed strike is positive even when it misses or is blocked. The website contains the [annotation workspace](https://strike.thomasou.com/workspace) and [recorded demo](https://strike.thomasou.com/demo); this notebook keeps the experiment record alongside the code.

**Current evidence:** a 38-window pilot, historical TSN training logs, and completed inference reproductions on 180 unreviewed windows from a separate frozen-ResNet development baseline, including a frame-order check on the 120 demo windows. There is **no independent test result** or established accuracy on unseen fights.

Run all cells with Python 3.11+ from this checkout. With the inference flag left at its default `False`, cells use the standard library to verify committed evidence and do not require PyTorch. The opt-in experiment reruns the local checkpoint; no cell trains a model or downloads footage.

## Research question

Can a five-frame appearance baseline distinguish committed strike attempts on held-out fights, and does foreground masking improve that comparison?

**Working hypothesis:** masking may reduce reliance on broadcast backgrounds, but it may also remove useful context or introduce segmentation errors. The direction and size of its effect remain unmeasured. The comparison changes the input representation while holding the reviewed windows, labels, fight assignments, encoder, sampling policy, and optimization settings fixed.

The outcome is a binary **window label**, not a count of strikes or a landed-strike judgment. A fight is the grouping unit for data separation; neighboring windows are dependent observations. Five sampled frames do not imply a fixed duration, so interval length and sampling policy must be recorded alongside the labels.

In [1]:
from pathlib import Path
import hashlib
import json
import math
import runpy
import statistics

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "model/export_notebook_history.py").is_file()
    and (path / "showcase/src/lib/recorded-demo.json").is_file()
)
history_path = ROOT / "showcase/src/lib/legacy-training.json"
demo_path = ROOT / "showcase/src/lib/recorded-demo.json"
history = json.loads(history_path.read_text())
demo = json.loads(demo_path.read_text())

for path in (history_path, demo_path):
    print(path.relative_to(ROOT))
    print(f"  SHA-256: {hashlib.sha256(path.read_bytes()).hexdigest()}")

showcase/src/lib/legacy-training.json
  SHA-256: 992bf6752ef59ac2f7db1354fe0a6c70da69015c49594ef049234761ac99b748
showcase/src/lib/recorded-demo.json
  SHA-256: 13b53ecad67124f6034907e947bad55e2da57dcf02f7b5c1e3752d1cff2d52f0


## Data and label definition

The original collection has **38 unique five-frame windows** (190 frames): 19 labeled strike and 19 neutral. Those labels predate the current attempt definition and need review.

| Original partition | Windows | Interpretation |
| :--- | ---: | :--- |
| Training | 30 | Development data |
| Validation | 6 | Inspected during model selection |
| Test | 2 | Both also occur in validation |
| Unused | 2 | Outside the original splits |

These memberships overlap; the rows must not be added as disjoint partitions. The legacy import now retains their memberships while assigning all examples to development. Fight identifiers and timestamps were not recorded. A sampled frame identifies Gaethje–Holloway; it does not establish the identity of every source clip.

**Attempt:** a committed strike, regardless of contact. **No attempt:** movement, guard adjustments, feints, or clinching alone. Keep occluded or ambiguous examples **uncertain** until reviewed; they are excluded from binary training labels. Contact outcome is a separate annotation.

Five frames need not contain a whole action. At an assumed 30 fps they occupy about 167 ms of playback, with 133 ms between the first and last sample. Actual source timing must come from the video; neither this assumption nor preview speed supplies missing timestamps. Window predictions also cannot be counted as unique strikes without event boundaries.

The [manifest and labeling reference](../../PROJECT_OVERVIEW.md) specifies source fight, interval, split, frame paths, and checksums. Assign entire fights to partitions before extracting windows, and keep test fights uninspected during model selection.

## Evidence available

| Evidence | What can be inspected | What it does not establish |
| :--- | :--- | :--- |
| Historical TSN run | Saved 20-epoch training and validation traces | Independent test performance; a comparison with the current baseline |
| Frozen-ResNet demo | Full 120-window inference reproduction and reversed-frame check; 600 input-frame hashes | Label accuracy, calibrated probabilities, or counts of unique strikes |
| Attributed ONE footage | Full 60-window inference reproduction; 300 local input-frame hashes and source provenance | An independently reviewed or demonstrably held-out fight benchmark |
| Raw versus masked inputs | Training, evaluation, and paired-comparison code | A measured benefit from masking; no matched result is recorded |

The default cells below recompute saved summaries and check their recorded identities; the demo frame files can also be checked directly. The expensive demo inference is opt-in, with its completed result saved alongside the notebook. Training and final evaluation require a separately reviewed dataset; the command examples at the end are an experiment procedure, not results from an executed benchmark.

## Historical TSN run

These are saved **MMAction2 TSN** metrics ([2](https://arxiv.org/abs/1608.00859)), not results for the newer frozen-backbone baseline. The exporter reads the original notebook output and records its SHA-256 and source cell indices. The following cell checks both the fingerprint and the complete exported history before displaying any metrics.

In [2]:
original_notebook = ROOT / history["source"]
assert hashlib.sha256(original_notebook.read_bytes()).hexdigest() == history["source_sha256"]
export_history = runpy.run_path(str(ROOT / "model/export_notebook_history.py"))["export_history"]
assert export_history(original_notebook) == history

print(f"Source: {history['source']}")
print(f"Output cells (zero-based): {history['cell_indices']}")
print(history["note"])
print()
print(f"{'Epoch':>5} {'Train loss':>12} {'Train accuracy':>16} {'Validation accuracy':>21}")
for epoch in history["epochs"]:
    print(f"{epoch['epoch']:5d} {epoch['train_loss']:12.4f} "
          f"{epoch['train_accuracy']:16.2%} {epoch['val_accuracy']:21.2%}")

Source: model/notebooks/TSN_Pipeline.ipynb
Output cells (zero-based): [6]
Historical validation logs, not a held-out benchmark. Both legacy test clips also appear in validation. No independent test result is recorded.

Epoch   Train loss   Train accuracy   Validation accuracy
    1       0.7090           46.67%               100.00%
    2       0.7002           60.00%               100.00%
    3       0.6881           73.33%                50.00%
    4       0.6810           63.33%                50.00%
    5       0.6676           83.33%                50.00%
    6       0.6525           90.00%                33.33%
    7       0.6375           86.67%                33.33%
    8       0.6192           96.67%                33.33%
    9       0.6028           93.33%                33.33%
   10       0.5843           96.67%                33.33%
   11       0.5645           96.67%                33.33%
   12       0.5441           96.67%                33.33%
   13       0.5227         

In [3]:
epochs = history["epochs"]
final = epochs[-1]
best_validation = max(epoch["val_accuracy"] for epoch in epochs)
best_epochs = [epoch["epoch"] for epoch in epochs if epoch["val_accuracy"] == best_validation]
print(f"Final epoch: {final['epoch']}")
print(f"Final train / validation accuracy: {final['train_accuracy']:.1%} / {final['val_accuracy']:.1%}")
print(f"Highest recorded validation accuracy: {best_validation:.1%}, epochs {best_epochs}")
print("Validation size: six clips. Independent test accuracy: not recorded.")

Final epoch: 20
Final train / validation accuracy: 100.0% / 50.0%
Highest recorded validation accuracy: 100.0%, epochs [1, 2]
Validation size: six clips. Independent test accuracy: not recorded.


The final epoch reaches **100% training accuracy and 50% validation accuracy**. Earlier 100% validation epochs describe six clips, not unseen-fight performance. The overlapping historical test set cannot resolve that limitation.

The original notebook uses ImageNet ResNet-50 weights without configuring the backbone as frozen. Its configuration does not fix the random seed, and its inherited test pipeline differs from validation. A downloaded Kinetics checkpoint is **not loaded** (`load_from=None`). The saved best checkpoint is from epoch 1, while the inference cell loads epoch 20.

The original [TSN notebook](TSN_Pipeline.ipynb) and [3D-CNN notebook](3DCNN_Pipeline.ipynb) remain historical artifacts. The latter is not a controlled comparison against the current baseline.

## Current appearance baseline

The implementation in [baseline.py](../baseline.py) freezes an ImageNet-1K V1 ResNet-50 ([1](https://arxiv.org/abs/1512.03385)), averages its frame features, and trains a binary linear head. Batch normalization in the encoder stays in evaluation mode.

| Setting | Value |
| :--- | :--- |
| Input | Five RGB frames |
| Preprocessing | Resize short edge to 256, center crop to 224, ImageNet normalization |
| Trainable parameters | Two-class linear head |
| Aggregation | Mean of the five frame feature vectors |
| Temporal order | Not modeled; permuting frames leaves the mean unchanged |

This can learn visual poses and context, but does not explicitly represent motion direction or timing. The new training pipeline records seed, preprocessing, dataset/source fingerprints, and checkpoint provenance. Research runs select on validation loss and evaluate the frozen choice separately on test data; development smoke runs are not benchmarks.

**Segmentation is an experiment.** SAM2 ([3](https://arxiv.org/abs/2408.00714)) overlays demonstrate a possible input variant, not an established improvement in detection. A raw/masked comparison needs the same reviewed examples, fight splits, seed, preprocessing, and training settings, with checkpoint selection confined to validation.

## Recorded demo predictions

The demo uses `recorded-demo.json`, not illustrative scores. It contains offline outputs from a **separate frozen-ResNet development checkpoint trained on the 38 pilot windows**. Its model scores are uncalibrated outputs; the JSON field named `confidence` does not make them calibrated confidence estimates.

The three displayed sequences have no reviewed ground-truth labels, source timestamps, or benchmark metrics. The visible scoreboard reads Herbert / Topuria, but the acquisition history and original video were not recorded. Cuts and replays may occur. Preview playback speed is not source timing.

The table below summarizes saved outputs. “Positive windows” means scores at or above the recorded threshold; it is **neither accuracy nor a count of strikes**.

In [4]:
model = demo["model"]
provenance = demo["provenance"]
threshold = model["threshold"]
assert demo["kind"] == "recorded_checkpoint_predictions"
assert provenance["groundTruthLabels"] is None
assert provenance["benchmarkMetrics"] is None

print(f"Architecture: {model['architecture']}")
print(f"Run purpose: {model['purpose']}; input variant: {model['variant']}")
print(f"Training windows: {model['trainingClipCount']}; selected epoch: {model['selectedEpoch']}")
print(f"Recorded threshold: {threshold:.2f}; frame-order sensitive: {model['frameOrderSensitive']}")
print()
print(f"{'Sequence':<12} {'Frames':>6} {'Windows':>8} {'Mean score':>11} {'Min':>8} {'Max':>8} {'Positive windows':>17}")
for name, clip in demo["clips"].items():
    scores = [prediction["confidence"] for prediction in clip["predictions"]]
    assert all(math.isfinite(score) and 0 <= score <= 1 for score in scores)
    positive = sum(score >= threshold for score in scores)
    print(f"{name:<12} {clip['totalFrames']:6d} {len(scores):8d} "
          f"{statistics.mean(scores):11.4f} {min(scores):8.4f} {max(scores):8.4f} {positive:17d}")
print(f"Total saved windows: {sum(len(clip['predictions']) for clip in demo['clips'].values())}")

Architecture: resnet50_frozen_frame_average_linear
Run purpose: development_smoke; input variant: raw
Training windows: 38; selected epoch: 20
Recorded threshold: 0.50; frame-order sensitive: False

Sequence     Frames  Windows  Mean score      Min      Max  Positive windows
knockdown       200       40      0.4142   0.0460   0.8780                13
exchange        200       40      0.4691   0.0056   0.9541                18
pressure        200       40      0.6367   0.1192   0.9540                29
Total saved windows: 120


### Check the displayed inputs

The next cell verifies every displayed frame against the hashes in the saved export and checks the five-frame window indices and threshold-derived labels. It does **not** rerun neural inference or validate label accuracy.

The export also preserves an earlier six-window reproduction check. The completed experiment below expands this to all 120 windows and adds frame reversal. Neither check establishes whether predictions correctly identify attempts.

In [5]:
verified_frames = 0
verified_windows = 0
for name, clip in demo["clips"].items():
    assert len(clip["frameSha256"]) == clip["totalFrames"]
    assert len(clip["predictions"]) * clip["windowSize"] == clip["totalFrames"]
    frame_dir = ROOT / "showcase/public" / clip["frameDir"].lstrip("/")
    for offset, expected_hash in enumerate(clip["frameSha256"]):
        file_index = offset + clip["frameFileIndexBase"]
        frame_path = frame_dir / (clip["framePattern"] % file_index)
        assert hashlib.sha256(frame_path.read_bytes()).hexdigest() == expected_hash, frame_path.name
        verified_frames += 1
    for index, prediction in enumerate(clip["predictions"]):
        start = index * clip["windowSize"]
        assert prediction["window"] == index
        assert prediction["startFrame"] == start
        assert prediction["endFrame"] == start + clip["windowSize"] - 1
        expected_label = "strike" if prediction["confidence"] >= threshold else "neutral"
        assert prediction["label"] == expected_label
        verified_windows += 1

print(f"Verified locally: {verified_frames} frame hashes and {verified_windows} window records.")
print(f"Recorded checkpoint SHA-256: {model['checkpointSha256']}")
print(f"Recorded training-manifest SHA-256: {model['trainingManifestSha256']}")
print(f"Previous export reproduction check: {provenance['reproducedWindowCount']} windows;")
print(f"maximum absolute score difference: {provenance['maxReproductionAbsoluteDifference']:.2e}.")
for filename, expected_hash in model["sourceSha256"].items():
    matches = hashlib.sha256((ROOT / "model" / filename).read_bytes()).hexdigest() == expected_hash
    print(f"Current source matches recorded source, {filename}: {matches}")

Verified locally: 600 frame hashes and 120 window records.
Recorded checkpoint SHA-256: 196f5a7fe8ea7c9916068aa20afc6e4fda67862dcf5991e53af2f1613957e4d2
Recorded training-manifest SHA-256: 289ff922ffd2dce394c1f85561675b1abaa04ea1cecbb50ac9dc957ef0d53091
Previous export reproduction check: 6 windows;
maximum absolute score difference: 5.96e-07.
Current source matches recorded source, baseline.py: True
Current source matches recorded source, data.py: True


## Completed inference reproduction and frame-order check

This experiment reruns **all 120 recorded windows (600 frames)** using the original saved development checkpoint and exactly the existing RGB preprocessing. It also runs the full network again with the five frames in each window reversed. The frame-average architecture predicts that reversal changes nothing except floating-point rounding; it cannot distinguish forward from backward motion using the same frames.

The saved [machine-readable report](../results/demo-reproduction.json) contains every original, reproduced, and reversed score, all input-frame hashes, checkpoint and code identities, package versions, execution settings, timestamps, and numerical differences. **Absolute tolerance: 1e-6**, fixed before execution. Threshold agreement measures implementation consistency, not accuracy. There are no reviewed labels, no accuracy/generalization estimate, and no raw-versus-masked comparison.

Completed on 2026-11-06 in **263.5 seconds on CPU**:

| Comparison | Windows | Maximum absolute score difference | Threshold changes |
| :--- | ---: | ---: | ---: |
| Saved versus rerun | 120 | 0 | 0 |
| Forward versus reversed frames | 120 | 1.19e-7 | 0 |

All windows meet the predeclared tolerance. Reversal supplies an observed sanity check of the architecture's inability to represent temporal order on these inputs.

The next two cells are opt-in. Leave `RUN_DEMO_INFERENCE = False` for a standard-library-only notebook run. To repeat inference, use the project virtual environment with the recorded package versions, supply the original local `runs/review-development` directory (checkpoint, configuration, and training record), set the flag to `True`, and choose a new output path. The checkpoint loader validates the original training hashes, architecture, preprocessing, and package versions before any inference; all 600 frame hashes are checked before and after execution. No weights or source video are downloaded. The following verification cell always checks the committed report without requiring those local model files.


In [6]:
RUN_DEMO_INFERENCE = False
DEMO_CHECKPOINT_RUN = ROOT / "runs/review-development"
DEMO_REPORT_OUTPUT = ROOT / "data-local/demo-reproduction.json"
if RUN_DEMO_INFERENCE:
    assert not DEMO_REPORT_OUTPUT.exists(), "Choose a new output path to preserve earlier reports."


In [7]:
def reproduce_demo(run_path: Path, output_path: Path) -> dict:
    from datetime import datetime, timezone
    import sys
    import time

    started_at = datetime.now(timezone.utc).isoformat()
    started = time.perf_counter()
    sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
    from PIL import Image
    import torch
    from torchvision.models import ResNet50_Weights
    from model.baseline import FrameAverageBaseline, configure_runtime, environment
    from model.predict import read_checkpoint

    absolute_tolerance = 1e-6
    reference_hash = hashlib.sha256(demo_path.read_bytes()).hexdigest()
    reference = json.loads(demo_path.read_text())
    checkpoint, config, checkpoint_hash = read_checkpoint(run_path)
    assert checkpoint_hash == reference["model"]["checkpointSha256"]
    assert config["manifest_sha256"] == reference["model"]["trainingManifestSha256"]
    assert config["preprocess"] == reference["model"]["preprocess"]
    assert config["threshold"] == reference["model"]["threshold"]
    assert config["selected_epoch"] == reference["model"]["selectedEpoch"]
    execution_environment = environment()
    for filename, expected in reference["model"]["sourceSha256"].items():
        assert execution_environment["source_sha256"][filename] == expected
    for package, version in reference["model"]["packages"].items():
        assert execution_environment[package] == version, package

    windows = []
    frame_inventory = []
    for name, clip in reference["clips"].items():
        assert clip["windowSize"] == 5
        assert len(clip["frameSha256"]) == clip["totalFrames"] == len(clip["predictions"]) * 5
        paths = [ROOT / "showcase/public" / clip["frameDir"].lstrip("/") /
                 (clip["framePattern"] % (index + clip["frameFileIndexBase"]))
                 for index in range(clip["totalFrames"])]
        frame_inventory.extend(zip(paths, clip["frameSha256"]))
        for index, prediction in enumerate(clip["predictions"]):
            start = index * 5
            assert (prediction["window"], prediction["startFrame"], prediction["endFrame"]) == (index, start, start + 4)
            windows.append({"sequence": name, "window": index, "start_frame": start, "end_frame": start + 4,
                            "frame_sha256": clip["frameSha256"][start:start + 5],
                            "recorded_probability_strike": prediction["confidence"], "paths": paths[start:start + 5]})
    for frame_path, expected in frame_inventory:
        assert hashlib.sha256(frame_path.read_bytes()).hexdigest() == expected, frame_path

    device = configure_runtime(config["seed"], "cpu", 4)
    network = FrameAverageBaseline("none")
    network.load_state_dict(checkpoint["model_state"], strict=True)
    network.to(device).eval()
    transform = ResNet50_Weights.IMAGENET1K_V1.transforms()
    batch_size = 4
    results = []
    for start in range(0, len(windows), batch_size):
        batch = windows[start:start + batch_size]
        tensors = []
        for window in batch:
            frames = []
            for frame_path in window["paths"]:
                with Image.open(frame_path) as frame:
                    frames.append(transform(frame.convert("RGB")))
            tensors.append(torch.stack(frames))
        inputs = torch.stack(tensors).to(device)
        with torch.inference_mode():
            forward = network(inputs).softmax(dim=1)[:, 1].cpu().tolist()
            reverse = network(inputs.flip(1)).softmax(dim=1)[:, 1].cpu().tolist()
        for window, normal_score, reverse_score in zip(batch, forward, reverse):
            results.append({key: value for key, value in window.items() if key != "paths"} |
                           {"reproduced_probability_strike": normal_score, "reversed_probability_strike": reverse_score})
        print(f"Inferred {min(start + batch_size, len(windows))}/{len(windows)} windows in both orders", flush=True)

    for frame_path, expected in frame_inventory:
        assert hashlib.sha256(frame_path.read_bytes()).hexdigest() == expected, frame_path
    assert hashlib.sha256(demo_path.read_bytes()).hexdigest() == reference_hash
    _, final_config, final_hash = read_checkpoint(run_path)
    assert final_hash == checkpoint_hash and final_config == config
    threshold = config["threshold"]
    def comparison_summary(rows: list[dict], left: str, right: str) -> dict:
        differences = [abs(row[left] - row[right]) for row in rows]
        return {"window_count": len(rows), "max_absolute_difference": max(differences),
                "mean_absolute_difference": statistics.mean(differences),
                "windows_within_tolerance": sum(value <= absolute_tolerance for value in differences),
                "threshold_disagreements": sum((row[left] >= threshold) != (row[right] >= threshold) for row in rows)}

    comparisons = {"reproduction": ("recorded_probability_strike", "reproduced_probability_strike"),
                   "frame_order_reversal": ("reproduced_probability_strike", "reversed_probability_strike")}
    summary = {name: comparison_summary(results, *fields) for name, fields in comparisons.items()}
    by_sequence = {name: {comparison: comparison_summary([row for row in results if row["sequence"] == name], *fields)
                         for comparison, fields in comparisons.items()} for name in reference["clips"]}
    notebook_path = ROOT / "model/notebooks/Research_Notes.ipynb"
    experiment_cell = next(cell for cell in json.loads(notebook_path.read_text())["cells"] if cell["id"] == "strike-demo-inference")
    report = {
        "schema_version": 1, "kind": "strike_demo_inference_reproduction", "status": "complete",
        "started_at": started_at, "completed_at": datetime.now(timezone.utc).isoformat(),
        "elapsed_seconds": time.perf_counter() - started,
        "reference": {"path": str(demo_path.relative_to(ROOT)), "sha256": reference_hash,
                      "exported_at": reference["exportedAt"]},
        "checkpoint": {"run": str(run_path.relative_to(ROOT)) if run_path.is_relative_to(ROOT) else str(run_path),
                       "sha256": checkpoint_hash, "purpose": config["purpose"], "architecture": config["architecture"],
                       "selected_epoch": config["selected_epoch"], "training_window_count": len(config["training_clip_ids"]),
                       "training_manifest_sha256": config["manifest_sha256"],
                       "configuration_sha256": hashlib.sha256(json.dumps(config, sort_keys=True, separators=(",", ":")).encode()).hexdigest(),
                       "configuration_file_sha256": hashlib.sha256((run_path / "config.json").read_bytes()).hexdigest(),
                       "training_record_sha256": hashlib.sha256((run_path / "training_metrics.json").read_bytes()).hexdigest()},
        "inputs": {"frame_root": "showcase/public", "frame_count": len(frame_inventory), "window_count": len(results),
                   "frames_per_window": 5, "ground_truth_labels": None, "benchmark_metrics": None,
                   "source_video": None, "source_timestamps": None,
                   "frame_hash_verification": "All frames before and after inference"},
        "execution": {"device": "cpu", "batch_size": batch_size, "threads": 4, "seed": config["seed"],
                      "deterministic_algorithms": True, "threshold": threshold, "absolute_tolerance": absolute_tolerance,
                      "preprocess": config["preprocess"], "reversal": "Reverse five input frames within each window; run the full network again"},
        "environment": execution_environment,
        "experiment": {"notebook": str(notebook_path.relative_to(ROOT)), "cell_id": experiment_cell["id"],
                       "cell_source_sha256": hashlib.sha256("".join(experiment_cell["source"]).encode()).hexdigest()},
        "summary": summary, "by_sequence": by_sequence, "windows": results,
        "interpretation": "Numerical reproduction and frame-order sanity check on unreviewed demo inputs. Threshold agreement is not accuracy. No generalization, calibration, strike-count, or masking-benefit estimate is made.",
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("x") as handle:
        handle.write(json.dumps(report, indent=2, allow_nan=False) + "\n")
    return report


if RUN_DEMO_INFERENCE:
    new_demo_report = reproduce_demo(DEMO_CHECKPOINT_RUN, DEMO_REPORT_OUTPUT)
    print(json.dumps(new_demo_report["summary"], indent=2))
else:
    print("Inference skipped. Reading the saved report below requires only the standard library.")


Inference skipped. Reading the saved report below requires only the standard library.


In [8]:
reproduction_path = ROOT / "model/results/demo-reproduction.json"
reproduction = json.loads(reproduction_path.read_text())
assert reproduction["kind"] == "strike_demo_inference_reproduction" and reproduction["status"] == "complete"
assert reproduction["reference"]["sha256"] == hashlib.sha256(demo_path.read_bytes()).hexdigest()
assert reproduction["checkpoint"]["sha256"] == model["checkpointSha256"]
assert reproduction["checkpoint"]["training_manifest_sha256"] == model["trainingManifestSha256"]
assert reproduction["checkpoint"]["training_window_count"] == model["trainingClipCount"]
assert reproduction["execution"]["preprocess"] == model["preprocess"]
assert reproduction["execution"]["threshold"] == threshold
assert reproduction["inputs"]["ground_truth_labels"] is None
assert reproduction["inputs"]["benchmark_metrics"] is None
assert reproduction["inputs"]["frame_count"] == verified_frames == 600
assert reproduction["inputs"]["window_count"] == verified_windows == 120
for filename, expected in model["sourceSha256"].items():
    assert reproduction["environment"]["source_sha256"][filename] == expected
    assert hashlib.sha256((ROOT / "model" / filename).read_bytes()).hexdigest() == expected
for package, expected in model["packages"].items():
    assert reproduction["environment"][package] == expected
notebook = json.loads((ROOT / reproduction["experiment"]["notebook"]).read_text())
experiment_cell = next(cell for cell in notebook["cells"] if cell["id"] == reproduction["experiment"]["cell_id"])
assert hashlib.sha256("".join(experiment_cell["source"]).encode()).hexdigest() == reproduction["experiment"]["cell_source_sha256"]

rows = reproduction["windows"]
expected_windows = [(name, prediction) for name, clip in demo["clips"].items() for prediction in clip["predictions"]]
assert len(rows) == len(expected_windows)
for row, (name, prediction) in zip(rows, expected_windows):
    assert (row["sequence"], row["window"], row["start_frame"], row["end_frame"]) == (name, prediction["window"], prediction["startFrame"], prediction["endFrame"])
    assert row["recorded_probability_strike"] == prediction["confidence"]
    assert row["frame_sha256"] == demo["clips"][name]["frameSha256"][row["start_frame"]:row["end_frame"] + 1]
    assert all(math.isfinite(row[field]) and 0 <= row[field] <= 1 for field in
               ("recorded_probability_strike", "reproduced_probability_strike", "reversed_probability_strike"))

absolute_tolerance = reproduction["execution"]["absolute_tolerance"]
assert absolute_tolerance == 1e-6
comparisons = {"reproduction": ("recorded_probability_strike", "reproduced_probability_strike"),
               "frame_order_reversal": ("reproduced_probability_strike", "reversed_probability_strike")}
for sequence, selected, summaries in [("all", rows, reproduction["summary"])] + [
    (name, [row for row in rows if row["sequence"] == name], reproduction["by_sequence"][name]) for name in demo["clips"]
]:
    for comparison, (left, right) in comparisons.items():
        differences = [abs(row[left] - row[right]) for row in selected]
        calculated = {"window_count": len(selected), "max_absolute_difference": max(differences),
                      "mean_absolute_difference": statistics.mean(differences),
                      "windows_within_tolerance": sum(value <= absolute_tolerance for value in differences),
                      "threshold_disagreements": sum((row[left] >= threshold) != (row[right] >= threshold) for row in selected)}
        assert summaries[comparison] == calculated
        print(f"{sequence:10} {comparison:21} n={len(selected):3d} max |delta|={max(differences):.2e} "
              f"within tolerance={calculated['windows_within_tolerance']:3d} threshold changes={calculated['threshold_disagreements']}")
print(f"Executed: {reproduction['started_at']} to {reproduction['completed_at']} ({reproduction['elapsed_seconds']:.1f} seconds)")
print(f"Report SHA-256: {hashlib.sha256(reproduction_path.read_bytes()).hexdigest()}")
print("Verified saved numerical results and provenance. No accuracy or masking benefit is measured.")


all        reproduction          n=120 max |delta|=0.00e+00 within tolerance=120 threshold changes=0
all        frame_order_reversal  n=120 max |delta|=1.19e-07 within tolerance=120 threshold changes=0
knockdown  reproduction          n= 40 max |delta|=0.00e+00 within tolerance= 40 threshold changes=0
knockdown  frame_order_reversal  n= 40 max |delta|=1.19e-07 within tolerance= 40 threshold changes=0
exchange   reproduction          n= 40 max |delta|=0.00e+00 within tolerance= 40 threshold changes=0
exchange   frame_order_reversal  n= 40 max |delta|=1.19e-07 within tolerance= 40 threshold changes=0
pressure   reproduction          n= 40 max |delta|=0.00e+00 within tolerance= 40 threshold changes=0
pressure   frame_order_reversal  n= 40 max |delta|=1.19e-07 within tolerance= 40 threshold changes=0
Executed: 2026-11-06T06:59:11.099804+00:00 to 2026-11-06T07:03:34.608711+00:00 (263.5 seconds)
Report SHA-256: 68a864b6006f12d2799a4ba9280930291031dbbf2ec57ab20bc0ddba15b21790
Verified saved n

## Separate footage: saved ONE inference reproduction

The [ONE reproduction report](../results/one-reproduction.json) records a completed rerun on **60 unreviewed one-second windows (300 prepared frames)** from the 120–180 second excerpt of ONE Championship's attributed Stamp Fairtex–Bi Nguyen full-fight video. It uses the same development checkpoint. All 60 probabilities match their preserved outputs exactly, with no threshold changes.

This supplies a different publicly attributed bout, but the legacy training collection lacks complete fight provenance, so disjointness from training is unproven. There are **zero reviewed labels**; neither the 41 positive predictions nor reproducibility is a measure of accuracy, generalization, or unique strikes.

The local run checked 300 frame hashes and 313 preserved files. Footage, prepared frames, and checkpoint weights remain outside Git. The cell below verifies the **committed report's internal consistency**, recorded code identity, and checkpoint agreement with the demo; it does not independently check those private frame bytes. The report retains the actual prediction command and local input/output paths for a repeat run with those artifacts. Frame extraction was not repeated.


In [9]:
one_path = ROOT / "model/results/one-reproduction.json"
one = json.loads(one_path.read_text())
assert one["kind"] == "strike_inference_reproduction" and one["status"] == "complete"
assert one["model"]["checkpoint_sha256"] == reproduction["checkpoint"]["sha256"]
assert one["model"]["threshold"] == threshold
assert one["scope"]["reviewed_labels"] == 0
assert all(one["scope"][metric] is None for metric in ("accuracy", "precision", "recall"))
assert not one["scope"]["training_performed"] and not one["scope"]["frame_extraction_repeated"]
assert not one["source"]["footage_included"]
assert one["scope"]["windows"] == len(one["windows"]) == 60
assert one["scope"]["frames"] == len(one["inputs"]["frame_sha256"]) == 300
assert one["inputs"]["frame_hashes_verified"] == 300
for filename, expected in one["execution"]["environment"]["source_sha256"].items():
    assert hashlib.sha256((ROOT / "model" / filename).read_bytes()).hexdigest() == expected
for package in ("python", "torch", "torchvision", "numpy", "pillow"):
    assert one["execution"]["environment"][package] == one["execution"]["recorded_environment"][package]

for index, row in enumerate(one["windows"]):
    assert row["clip_id"] == f"one-stamp-nguyen-{index:03d}"
    assert (row["start_time_s"], row["end_time_s"]) == (index, index + 1)
    assert row["annotation_label"] == "unreviewed"
    for prefix in ("recorded", "reproduced"):
        probability = row[f"{prefix}_probability_strike"]
        assert math.isfinite(probability) and 0 <= probability <= 1
        assert row[f"{prefix}_threshold_prediction"] == int(probability >= threshold)
    assert row["absolute_difference"] == abs(row["recorded_probability_strike"] - row["reproduced_probability_strike"])
    for frame_index in range(1, 6):
        digest = one["inputs"]["frame_sha256"][f"frames/clip_{index:06d}/img_{frame_index:05d}.png"]
        assert len(digest) == 64 and all(character in "0123456789abcdef" for character in digest)

one_scores = [row["reproduced_probability_strike"] for row in one["windows"]]
one_differences = [row["absolute_difference"] for row in one["windows"]]
expected_summary = {
    "exact_probability_matches": sum(value == 0 for value in one_differences),
    "max_absolute_probability_difference": max(one_differences),
    "mean_absolute_probability_difference": statistics.mean(one_differences),
    "threshold_prediction_matches": sum(row["recorded_threshold_prediction"] == row["reproduced_threshold_prediction"] for row in one["windows"]),
    "positive_predictions_at_threshold": sum(score >= threshold for score in one_scores),
    "min_score": min(one_scores), "max_score": max(one_scores),
    "mean_score": statistics.mean(one_scores), "median_score": statistics.median(one_scores),
}
for field, expected in expected_summary.items():
    assert one["comparison"][field] == expected, field
assert one["comparison"]["exact_probability_matches"] == 60
assert one["comparison"]["prediction_files_byte_identical"] == (one["outputs"]["recorded_predictions_sha256"] == one["outputs"]["reproduced_predictions_sha256"])
print(f"Saved ONE report: {len(one_scores)} windows; exact score matches: {expected_summary['exact_probability_matches']}; max |delta|: {max(one_differences):.2e}")
print(f"Completed: {one['completed_at']}; reviewed labels: {one['scope']['reviewed_labels']}")
print(f"Positive model predictions: {expected_summary['positive_predictions_at_threshold']}; these are not strike counts or accuracy.")
print(f"Report SHA-256: {hashlib.sha256(one_path.read_bytes()).hexdigest()}")
print("Verified saved report consistency. The original local frame and video bytes are not verified by this cell.")


Saved ONE report: 60 windows; exact score matches: 60; max |delta|: 0.00e+00
Completed: 2026-11-06T06:58:02.785587+00:00; reviewed labels: 0
Positive model predictions: 41; these are not strike counts or accuracy.
Report SHA-256: 2ede17d9bd22a5c07c1023595fdc4983f424a7688c8136947abfc53b82a66b9f
Verified saved report consistency. The original local frame and video bytes are not verified by this cell.


## Planned comparison and reporting

This is a proposed design for the next experiment, not a preregistration or a completed result. Finalize it on development and validation data before inspecting test predictions.

| Decision | Planned treatment |
| :--- | :--- |
| Comparison | Raw frames versus their paired masked frames, trained separately |
| Fixed inputs | Same reviewed manifest, fight assignments, five sampled frames, and labels |
| Fixed training | Same frozen encoder, preprocessing, seed, optimizer, and training budget |
| Checkpoint selection | Lowest validation loss within each run; selected epochs may differ |
| Primary comparison | Masked minus raw balanced accuracy at the recorded threshold of 0.5 |
| Supporting results | Average precision, precision/recall/F1, log loss, Brier score, and confusion matrices |
| Scope of the estimate | The reserved fights and the documented window-sampling policy |

Report class counts, numbers of windows and fights, and each fight's metrics alongside the pooled result. Balanced accuracy requires both classes; leave it undefined for a single-class fight rather than substituting zero. Pooled metrics weight windows equally, so fights with more windows have more influence. Class prevalence also affects precision and average precision; a balanced collection of selected windows does not represent the prevalence in an entire fight.

The comparison tool checks matched runs and reports which windows change correctness. It produces descriptive differences, not confidence intervals or significance tests. Any later uncertainty analysis must account for dependence within fights; treating neighboring windows as independent would overstate the evidence. Repeat any selected seed comparisons under the same frozen protocol, and report all runs rather than choosing the best test result.

## What remains unmeasured

- **New fights:** independently reviewed footage with fight-level separation and an untouched test partition. More neighboring windows from the same sequence do not establish generalization.
- **Hard negatives and missed attempts:** feints, clinch breaks, fast movement, blocked strikes, and misses need separate evaluation slices.
- **Occlusion and boundaries:** incomplete visibility and attempts crossing windows need reviewed event timestamps before reporting event-level recall or counts.
- **Masking:** a paired experiment is required before attributing any improvement to segmentation.

These are open evaluation questions, not measured failure rates. No demo score, training accuracy, or earlier validation peak answers them.

## Reproduce and extend

All commands below run from the repository root. See the [README](../../README.md) for Python dependencies and complete setup. The default verification cells above require no model dependencies. The opt-in demo inference cell needs the recorded environment and local checkpoint.

Regenerate the historical JSON from the saved TSN notebook output:

```bash
python model/export_notebook_history.py
```

For new footage, label windows in the workspace and export its annotation backup. Keep source video outside Git. Prepare the frames and score them using an existing completed raw model run:

```bash
python -m model.prepare --annotations annotations.json --video /path/to/footage.mp4 --output data-local/reviewed
python -m model.predict --prepared data-local/reviewed --run runs/your-raw-run --output runs/workspace-review
```

Import `runs/workspace-review/predictions.json` into the workspace. The browser reviews footage and scores; inference runs locally in Python. Predictions do not replace manual labels. Use new output directories to preserve prior results.

For a research experiment, curate one manifest with reviewed labels, source provenance, paired raw and masked frames, and whole fights assigned to train, validation, and test before training. Use the same environment for both runs and new output paths throughout.

Audit both input variants, then train the matched models:

```bash
python -m model.data audit --manifest data-local/reviewed.json --data-root /path/to/frames --variant raw --output runs/raw-audit.json
python -m model.data audit --manifest data-local/reviewed.json --data-root /path/to/frames --variant masked --output runs/masked-audit.json
python -m model.train --manifest data-local/reviewed.json --data-root /path/to/frames --output runs/raw --variant raw --seed 42
python -m model.train --manifest data-local/reviewed.json --data-root /path/to/frames --output runs/masked --variant masked --seed 42
python -m model.compare --raw runs/raw --masked runs/masked --split validation --output runs/validation-comparison.json
```

Select the design on validation. After freezing the design, checkpoints, and reporting plan, evaluate the same reserved fights for both models:

```bash
python -m model.evaluate --run runs/raw --manifest data-local/reviewed.json --data-root /path/to/frames
python -m model.evaluate --run runs/masked --manifest data-local/reviewed.json --data-root /path/to/frames
python -m model.compare --raw runs/raw --masked runs/masked --split test --output runs/test-comparison.json
```

Each run records its configuration, manifest, audit, training history, selected checkpoint, and per-window predictions. Test outputs live under each run's `evaluation/` directory. The comparison JSON records metric differences and paired correctness outcomes; retain it with both run directories.

The current pilot cannot supply that independent test set. These commands describe the supported workflow; this notebook has not run new training or test evaluation.

## References

1. Kaiming He et al. (2015). [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385). Backbone used by the current appearance baseline.
2. Limin Wang et al. (2016). [Temporal Segment Networks: Towards Good Practices for Deep Action Recognition](https://arxiv.org/abs/1608.00859). Method underlying the historical TSN experiment.
3. Nikhila Ravi et al. (2024). [SAM 2: Segment Anything in Images and Videos](https://arxiv.org/abs/2408.00714). Segmentation method referenced by the overlay scripts; a raw/masked performance comparison is not yet reported here.